# 🚚 ETL Bronze → Silver — Dataset 2 : E-Commerce Shipping
## Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce

---

| Champ | Valeur |
|---|---|
| **Dataset** | E-Commerce Shipping |
| **Volume initial (Bronze)** | ~10 999 lignes |
| **Colonnes** | 12 colonnes |
| **Fichier source** | `ecommerce_shipping.csv` |
| **Sortie Silver** | `ecommerce_shipping_silver.parquet` |
| **Journal** | `DECISIONS.md` — DEC-012 à DEC-019 |

---

### Périmètre métier de ce dataset
| Sous-thème | Usage |
|---|---|
| **ST1** | `Reached.on.Time_Y.N` → impact retard livraison sur comportement |
| **ST2** | `Customer_rating`, `Prior_purchases` → corrélation satisfaction × segment RFM |
| **ST4** | `Discount_offered`, `Cost_of_the_Product`, `Weight_in_gms`, `Mode_of_Shipment` → marge nette |

## 0. Montage Google Drive & Imports

In [ ]:
# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('ETL_SHIPPING')

print('✅ Imports OK')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

## 1. Configuration des chemins

In [ ]:
# ─── Chemins du projet ───────────────────────────────────────────────────────
BASE_PATH   = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce'
RAW_PATH    = os.path.join(BASE_PATH, 'Data', 'raw')
SILVER_PATH = os.path.join(BASE_PATH, 'Data', 'silver')

os.makedirs(SILVER_PATH, exist_ok=True)

FILE_IN  = os.path.join(RAW_PATH,    'ecommerce_shipping.csv')
FILE_OUT = os.path.join(SILVER_PATH, 'ecommerce_shipping_silver.parquet')

print(f'📂 Dossier source  : {RAW_PATH}')
print(f'📂 Dossier Silver  : {SILVER_PATH}')
print(f'📄 Fichier source  : {FILE_IN}')
assert os.path.exists(FILE_IN), f'❌ Fichier introuvable : {FILE_IN}'

## 2. Chargement Bronze

In [ ]:
logger.info('=== CHARGEMENT BRONZE — E-Commerce Shipping ===')

df_bronze = pd.read_csv(FILE_IN, encoding='utf-8')

VOLUME_INITIAL = len(df_bronze)
logger.info(f'Volume initial (Bronze) : {VOLUME_INITIAL:,} lignes')

print(f'📐 Shape : {df_bronze.shape}')
print(f'\n📋 Colonnes :')
print(df_bronze.dtypes.to_string())
print(f'\n📊 Aperçu :')
df_bronze.head(3)

In [ ]:
# Rapport de qualité initial
print('\n📋 RAPPORT QUALITÉ INITIAL (Bronze)')
print('='*65)
for col in df_bronze.columns:
    n_null = df_bronze[col].isna().sum()
    pct    = n_null / VOLUME_INITIAL * 100
    n_uniq = df_bronze[col].nunique()
    dtype  = df_bronze[col].dtype
    print(f'  {col:<35} | nulls: {n_null:>5,} ({pct:4.1f}%) | uniques: {n_uniq:>5,} | dtype: {dtype}')

print(f'\n  Doublons stricts : {df_bronze.duplicated().sum():,}')

## 3. Pipeline de Nettoyage Bronze → Silver

In [ ]:
# Compteur de pertes
losses = {}
df = df_bronze.copy()

def log_loss(dec_id, description, before, after, note=''):
    lost = before - after
    pct  = lost / VOLUME_INITIAL * 100
    losses[dec_id] = {'description': description, 'lost': lost, 'pct': pct}
    logger.info(f'[{dec_id}] {description} → -{lost:,} lignes ({pct:.2f}%)')
    if note:
        logger.info(f'         Note : {note}')
    return after

### DEC-012 — Suppression des doublons stricts

In [ ]:
# ── DEC-012 : Doublons stricts ───────────────────────────────────────────────
before = len(df)
n_dupes = df.duplicated().sum()
print(f'  Doublons stricts détectés : {n_dupes:,}')

df = df.drop_duplicates(keep='first')
after = len(df)
log_loss('DEC-012', 'Doublons stricts supprimés', before, after)

### DEC-013 — Validation et catégorisation de Discount_offered

In [ ]:
# ── DEC-013 : Discount_offered ──────────────────────────────────────────────
print(f'  Discount_offered — statistiques :')
print(df['Discount_offered'].describe().to_string())

# Vérification des bornes [0, 100]
before = len(df)
n_out_of_range = ((df['Discount_offered'] < 0) | (df['Discount_offered'] > 100)).sum()
print(f'\n  Valeurs hors [0, 100] : {n_out_of_range:,}')

df = df[(df['Discount_offered'] >= 0) & (df['Discount_offered'] <= 100)].copy()
after = len(df)
log_loss('DEC-013', 'Discount_offered hors [0,100] exclu', before, after)

# Catégorisation en tranches (ST4)
bins   = [-1, 0, 10, 25, 50, 100]
labels = ['0%_prix_plein', '1-10%_leger', '11-25%_significatif', '26-50%_fort', '>50%_liquidation']
df['discount_category'] = pd.cut(df['Discount_offered'], bins=bins, labels=labels)

print('\n  Distribution par tranche de remise :')
print(df['discount_category'].value_counts().sort_index().to_string())

### DEC-014 — Traitement des valeurs aberrantes de Cost_of_the_Product

In [ ]:
# ── DEC-014 : Cost_of_the_Product ───────────────────────────────────────────
print(f'  Cost_of_the_Product — statistiques :')
print(df['Cost_of_the_Product'].describe().to_string())

# Exclusion coûts <= 0
before = len(df)
n_invalid = (df['Cost_of_the_Product'] <= 0).sum()
print(f'\n  Coûts <= 0 : {n_invalid:,}')

df = df[df['Cost_of_the_Product'] > 0].copy()
after = len(df)
log_loss('DEC-014', 'Cost_of_the_Product <= 0 exclus', before, after)

# Flag outliers hauts (IQR × 3 — produits premium légitimes)
Q1_cost = df['Cost_of_the_Product'].quantile(0.25)
Q3_cost = df['Cost_of_the_Product'].quantile(0.75)
IQR_cost = Q3_cost - Q1_cost
UPPER_cost = Q3_cost + 3 * IQR_cost

df['is_high_cost'] = df['Cost_of_the_Product'] > UPPER_cost
print(f'  Seuil outlier haut (IQR×3) : {UPPER_cost:.2f}')
print(f'  Produits premium flaggés   : {df["is_high_cost"].sum():,}')

### DEC-015 — Validation et catégorisation de Weight_in_gms

In [ ]:
# ── DEC-015 : Weight_in_gms ─────────────────────────────────────────────────
print(f'  Weight_in_gms — statistiques :')
print(df['Weight_in_gms'].describe().to_string())

# Exclusion poids <= 0
before = len(df)
n_invalid_w = (df['Weight_in_gms'] <= 0).sum()
print(f'\n  Poids <= 0 : {n_invalid_w:,}')

if n_invalid_w > 0:
    df = df[df['Weight_in_gms'] > 0].copy()
after = len(df)
log_loss('DEC-015', 'Weight_in_gms <= 0 exclus', before, after)

# Création de weight_category (tranches en grammes)
bins_w   = [0, 1000, 2000, 4000, 6000, float('inf')]
labels_w = ['<1kg', '1-2kg', '2-4kg', '4-6kg', '>6kg']
df['weight_category'] = pd.cut(df['Weight_in_gms'], bins=bins_w, labels=labels_w)

print('\n  Distribution par tranche de poids :')
print(df['weight_category'].value_counts().sort_index().to_string())

### DEC-016 — Validation de Reached.on.Time_Y.N

In [ ]:
# ── DEC-016 : Reached.on.Time_Y.N ───────────────────────────────────────────
print(f'  Valeurs uniques Reached.on.Time_Y.N : {df["Reached.on.Time_Y.N"].unique()}')
print(f'  Distribution :\n{df["Reached.on.Time_Y.N"].value_counts().to_string()}')

# Vérification bornes {0, 1}
before = len(df)
n_invalid_binary = (~df['Reached.on.Time_Y.N'].isin([0, 1])).sum()
print(f'\n  Valeurs hors {{0, 1}} : {n_invalid_binary:,}')

if n_invalid_binary > 0:
    df = df[df['Reached.on.Time_Y.N'].isin([0, 1])].copy()
after = len(df)
log_loss('DEC-016', 'Reached.on.Time hors {0,1} exclus', before, after)

# Renommage snake_case
df.rename(columns={'Reached.on.Time_Y.N': 'delivered_on_time'}, inplace=True)
df['delivered_on_time'] = df['delivered_on_time'].astype(int)

# Label lisible
df['delivery_status'] = df['delivered_on_time'].map({1: 'On_Time', 0: 'Late'})

print(f'  ✅ Renommé en delivered_on_time')
print(f'  Distribution delivery_status :')
print(df['delivery_status'].value_counts().to_string())

### DEC-017 — Validation de Customer_rating

In [ ]:
# ── DEC-017 : Customer_rating ───────────────────────────────────────────────
print(f'  Valeurs uniques Customer_rating : {sorted(df["Customer_rating"].unique())}')
print(f'  Distribution :\n{df["Customer_rating"].value_counts().sort_index().to_string()}')

# Vérification bornes {1, 2, 3, 4, 5}
before = len(df)
VALID_RATINGS = {1, 2, 3, 4, 5}
n_invalid_rating = (~df['Customer_rating'].isin(VALID_RATINGS)).sum()
print(f'\n  Notes hors [1-5] : {n_invalid_rating:,}')

if n_invalid_rating > 0:
    df = df[df['Customer_rating'].isin(VALID_RATINGS)].copy()
after = len(df)
log_loss('DEC-017', 'Customer_rating hors [1-5] exclus', before, after)

# Label de satisfaction
df['satisfaction_label'] = df['Customer_rating'].map({
    1: 'Très insatisfait',
    2: 'Insatisfait',
    3: 'Neutre',
    4: 'Satisfait',
    5: 'Très satisfait'
})
print(f'  ✅ satisfaction_label créé')

### DEC-018 — Standardisation de Product_importance

In [ ]:
# ── DEC-018 : Product_importance ────────────────────────────────────────────
print(f'  Valeurs uniques Product_importance : {df["Product_importance"].unique()}')

# 1. Normalisation UPPER CASE
df['Product_importance'] = df['Product_importance'].str.strip().str.capitalize()

# 2. Vérification des valeurs valides
before = len(df)
VALID_IMPORTANCE = {'Low', 'Medium', 'High'}
n_invalid_imp = (~df['Product_importance'].isin(VALID_IMPORTANCE)).sum()
print(f'  Valeurs hors {{Low, Medium, High}} : {n_invalid_imp:,}')

if n_invalid_imp > 0:
    df = df[df['Product_importance'].isin(VALID_IMPORTANCE)].copy()
after = len(df)
log_loss('DEC-018', 'Product_importance invalide exclus', before, after)

# 3. Encodage ordinal
importance_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['product_importance_score'] = df['Product_importance'].map(importance_map).astype(int)

losses['DEC-018'] = {'description': 'Product_importance encodé ordinal (0 perte)', 'lost': 0, 'pct': 0.0}
print(f'\n  Distribution product_importance_score :')
print(df['product_importance_score'].value_counts().sort_index().to_string())

### DEC-019 — Grille tarifaire hypothétique des frais de port

In [ ]:
# ── DEC-019 : Grille tarifaire livraison ─────────────────────────────────────
# Hypothèse documentée dans DECISIONS.md
# Source de référence : tarifs publics UK (Royal Mail, DPD, FedEx, Hermes/Evri, DHL)

SHIPPING_GRID = {
    ('Ship',   '<1kg') : 2.50,
    ('Ship',   '1-2kg'): 3.95,
    ('Ship',   '2-4kg'): 5.50,
    ('Ship',   '4-6kg'): 7.50,
    ('Ship',   '>6kg') : 10.00,
    ('Flight', '<1kg') : 8.00,
    ('Flight', '1-2kg'): 12.00,
    ('Flight', '2-4kg'): 18.00,
    ('Flight', '4-6kg'): 25.00,
    ('Flight', '>6kg') : 35.00,
    ('Road',   '<1kg') : 1.80,
    ('Road',   '1-2kg'): 2.80,
    ('Road',   '2-4kg'): 4.20,
    ('Road',   '4-6kg'): 6.00,
    ('Road',   '>6kg') : 8.50,
}

# Affichage de la grille
print('  ⚠️  HYPOTHÈSE DOCUMENTÉE (DEC-019) — Grille tarifaire frais de port')
print('  ' + '-'*60)
print(f'  {"Mode":<10} {"Tranche":<10} {"Coût (£)":>10}  {"Source"}')
print('  ' + '-'*60)
ref_sources = {
    ('Ship',   '<1kg') : 'Royal Mail Large Letter',
    ('Ship',   '1-2kg'): 'Royal Mail Parcel S',
    ('Ship',   '2-4kg'): 'DPD Standard',
    ('Ship',   '4-6kg'): 'DPD Standard',
    ('Ship',   '>6kg') : 'DPD Express',
    ('Flight', '<1kg') : 'FedEx Intl Economy',
    ('Flight', '1-2kg'): 'FedEx Intl Economy',
    ('Flight', '2-4kg'): 'FedEx Intl Priority',
    ('Flight', '4-6kg'): 'FedEx Intl Priority',
    ('Flight', '>6kg') : 'FedEx Intl Priority',
    ('Road',   '<1kg') : 'Hermes Standard',
    ('Road',   '1-2kg'): 'Hermes Standard',
    ('Road',   '2-4kg'): 'DHL Road',
    ('Road',   '4-6kg'): 'DHL Road',
    ('Road',   '>6kg') : 'DHL Road',
}
for (mode, tranche), cost in SHIPPING_GRID.items():
    print(f'  {mode:<10} {tranche:<10} £{cost:>7.2f}   {ref_sources[(mode, tranche)]}')

# Application de la grille
df['weight_category_str'] = df['weight_category'].astype(str)
df['estimated_shipping_cost'] = df.apply(
    lambda row: SHIPPING_GRID.get(
        (row['Mode_of_Shipment'], row['weight_category_str']),
        np.nan
    ),
    axis=1
)
df.drop(columns=['weight_category_str'], inplace=True)

n_missing_cost = df['estimated_shipping_cost'].isna().sum()
losses['DEC-019'] = {'description': 'Grille tarifaire appliquée (0 perte)', 'lost': 0, 'pct': 0.0}
logger.info(f'[DEC-019] estimated_shipping_cost calculé — {n_missing_cost} valeurs manquantes')
print(f'\n  ✅ estimated_shipping_cost créé')
print(f'  Valeurs manquantes : {n_missing_cost:,}')
print(f'  Coût moyen estimé  : £{df["estimated_shipping_cost"].mean():.2f}')

### Feature Engineering complémentaire (ST4)

In [ ]:
# ── Feature Engineering ST4 : Marge nette ────────────────────────────────────
# Toutes les colonnes nécessaires au calcul de marge dans dbt Gold

# Marge brute = pas de TotalRevenue ici (pas de colonne prix de vente dans ce dataset)
# On crée les inputs nécessaires

# Montant de la remise en valeur (en % du coût — proxy)
df['discount_amount_proxy'] = (
    df['Cost_of_the_Product'] * df['Discount_offered'] / 100
).round(2)

# Coût net après remise (du point de vue retailer : ce que le client paie moins)
df['net_cost_after_discount'] = (
    df['Cost_of_the_Product'] - df['discount_amount_proxy']
).round(2)

# Marge nette proxy : net_cost - shipping_cost
df['margin_proxy'] = (
    df['net_cost_after_discount'] - df['estimated_shipping_cost'].fillna(0)
).round(2)

print('  ✅ Features ST4 créés :')
print('    + discount_amount_proxy')
print('    + net_cost_after_discount')
print('    + margin_proxy')
print(f'\n  Marge proxy moyenne : £{df["margin_proxy"].mean():.2f}')
print(f'  Marge proxy médiane : £{df["margin_proxy"].median():.2f}')
print(f'  Commandes en perte  : {(df["margin_proxy"] < 0).sum():,}')

### Renommage des colonnes (snake_case)

In [ ]:
# ── Renommage colonnes en snake_case ─────────────────────────────────────────
RENAME_MAP = {
    'ID'                    : 'order_id',
    'Warehouse_block'       : 'warehouse_block',
    'Mode_of_Shipment'      : 'mode_of_shipment',
    'Customer_care_calls'   : 'customer_care_calls',
    'Customer_rating'       : 'customer_rating',
    'Cost_of_the_Product'   : 'cost_of_product',
    'Prior_purchases'       : 'prior_purchases',
    'Product_importance'    : 'product_importance',
    'Gender'                : 'gender',
    'Discount_offered'      : 'discount_pct',
    'Weight_in_gms'         : 'weight_gms',
}

# Renommer seulement les colonnes qui existent
rename_actual = {k: v for k, v in RENAME_MAP.items() if k in df.columns}
df.rename(columns=rename_actual, inplace=True)

print('  ✅ Colonnes renommées en snake_case :')
for old, new in rename_actual.items():
    print(f'    {old} → {new}')

## 4. Bilan Cumulé des Pertes

In [ ]:
print('\n' + '='*65)
print('  BILAN CUMULÉ DES PERTES — E-Commerce Shipping')
print('='*65)
print(f'  Volume initial (Bronze) : {VOLUME_INITIAL:>10,}')
print()

total_lost = 0
for dec_id, info in losses.items():
    print(f'  {dec_id} | {info["description"]:<50} | -{info["lost"]:>5,} ({info["pct"]:5.2f}%)')
    total_lost += info['lost']

pct_total = total_lost / VOLUME_INITIAL * 100
print()
print(f'  Volume Silver final      : {len(df):>10,}')
print(f'  Perte totale             : {pct_total:.2f}% ' + ('✅' if pct_total < 5 else '❌'))
print('='*65)

## 5. Schéma final Silver

In [ ]:
# Réorganisation des colonnes Silver
SILVER_COLS_ORDER = [
    # Identifiant
    'order_id',
    # Logistique (ST1, ST4)
    'warehouse_block', 'mode_of_shipment', 'weight_gms', 'weight_category',
    'delivered_on_time', 'delivery_status',
    # Client (ST2)
    'gender', 'customer_rating', 'satisfaction_label', 'prior_purchases',
    'customer_care_calls',
    # Produit (ST4)
    'product_importance', 'product_importance_score', 'is_high_cost',
    # Financier (ST4)
    'cost_of_product', 'discount_pct', 'discount_category', 'discount_amount_proxy',
    'estimated_shipping_cost', 'net_cost_after_discount', 'margin_proxy',
]

# Garder uniquement les colonnes qui existent
existing_cols = [c for c in SILVER_COLS_ORDER if c in df.columns]
df_silver = df[existing_cols].copy()

print('📋 Schéma Silver final :')
print(df_silver.dtypes.to_string())
print(f'\n📐 Shape Silver : {df_silver.shape}')
print(f'\n  Nulls restants :')
null_remaining = df_silver.isna().sum()
if null_remaining.sum() > 0:
    print(null_remaining[null_remaining > 0].to_string())
else:
    print('  Aucun null')

## 6. Sauvegarde Parquet

In [ ]:
# Conversion des colonnes catégorielles pour Parquet
for col in ['discount_category', 'weight_category', 'delivery_status', 'satisfaction_label']:
    if col in df_silver.columns:
        df_silver[col] = df_silver[col].astype(str)

df_silver.to_parquet(FILE_OUT, index=False, engine='pyarrow')
logger.info(f'✅ ecommerce_shipping_silver sauvegardé : {FILE_OUT}')

print(f'  💾 {FILE_OUT}')
print(f'     {len(df_silver):,} lignes | {os.path.getsize(FILE_OUT)/1024:.1f} KB')

## 7. Vérification de reproductibilité

In [ ]:
df_check = pd.read_parquet(FILE_OUT)

assert len(df_check) == len(df_silver), '❌ Erreur de reproductibilité : volumes différents'
assert list(df_check.columns) == list(df_silver.columns), '❌ Erreur de reproductibilité : colonnes différentes'

print('✅ Vérification de reproductibilité : OK')
print(f'   {len(df_check):,} lignes relues depuis le Parquet')

## 8. Statistiques finales Silver

In [ ]:
print('\n📊 STATISTIQUES SILVER — E-Commerce Shipping')
print('='*55)
print(f'  Commandes           : {len(df_silver):,}')
if 'mode_of_shipment' in df_silver.columns:
    print(f'\n  Distribution mode de livraison :')
    print(df_silver['mode_of_shipment'].value_counts().to_string())
print(f'\n  Taux de livraison à temps : {df_silver["delivered_on_time"].mean()*100:.1f}%')
print(f'  Note moyenne clients      : {df_silver["customer_rating"].mean():.2f} / 5')
print(f'  Remise moyenne            : {df_silver["discount_pct"].mean():.1f}%')
print(f'  Coût produit moyen        : £{df_silver["cost_of_product"].mean():.2f}')
print(f'  Frais de port estimés moy : £{df_silver["estimated_shipping_cost"].mean():.2f}')
print(f'  Marge proxy moyenne       : £{df_silver["margin_proxy"].mean():.2f}')
print(f'  Commandes en perte        : {(df_silver["margin_proxy"] < 0).sum():,}')

In [ ]:
print(f'\n✅ Pipeline ETL Bronze → Silver Shipping terminé avec succès.')
print(f'   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'\n   Fichier produit :')
print(f'   → {FILE_OUT}')